<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/homework/WRIT20833_HW2_2026_ANSWER_KEY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 2 — Term Frequency: Whose Words Win? — INSTRUCTOR ANSWER KEY

**WRIT 20833 — Intro to Coding in the Humanities**

> Solutions are one valid approach; students may reach the same result differently. Teaching
> notes appear under each solution. This key **runs top-to-bottom** against the two corpora in
> `notebooks/data/`. Counts below reflect those exact files (2026-06).

---

In [ ]:
# SETUP — run this cell first. (You're not expected to read every line yet — that's fine.)
import re
from collections import Counter
import os, urllib.request

def split_into_words(any_chunk_of_text):
    lowercase_text = any_chunk_of_text.lower()
    # Split on any run of non-letter/number characters (spaces, punctuation, newlines).
    return re.split(r"\W+", lowercase_text)

# The same long stopwords list you met in HW1 (A6): tiny function words that carry
# little meaning on their own. We filter these out to find the "meaningful" words.
stopwords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now", "ve", "ll", "amp"]

def load_text(filename):
    """Load a corpus. Checks this folder and notebooks/data/, then falls back to
    downloading from the course repo (works in Colab, and locally after merge)."""
    for path in (filename, os.path.join("notebooks", "data", filename),
                 os.path.join("data", filename), os.path.join("..", "data", filename)):
        if os.path.exists(path):
            return open(path, encoding="utf-8").read()
    url = "https://raw.githubusercontent.com/TCU-DCDA/WRIT20833_2026/main/notebooks/data/" + filename
    return urllib.request.urlopen(url).read().decode("utf-8")

# Two real corpora about the 2025 Texas Ten Commandments-in-schools law:
comments_text = load_text("tc_youtube_comments.txt")    # the public's voice (93 YouTube comments)
constitution_text = load_text("us_constitution.txt")    # the document the public keeps invoking
print("Setup complete. Both corpora loaded.")

## Part A — From words to counts (5 exercises)

**A1 — How big is each corpus?**

In [ ]:
# A1 — solution
for name, text in [("CONSTITUTION", constitution_text), ("COMMENTS", comments_text)]:
    words = split_into_words(text)
    print(name, "-> total:", len(words), " distinct:", len(set(words)))
# CONSTITUTION -> total: ~4556  distinct: ~937   (includes one empty "" token from the split)
# COMMENTS     -> total: ~2282  distinct: ~748

> **Teaching note:** Goal: `len` (total) vs. `len(set(...))` (distinct), reused from HW1 C2.
> Note the split can yield a single empty string `""` (leading/trailing punctuation) — that's why
> A3 filters `if w` as well as stopwords. The Constitution is longer but the comments are
> lexically *denser* (more distinct words per total) — vernacular variety vs. legal repetition.

**A2 — The raw count problem.**

In [ ]:
# A2 — solution
all_counts = Counter(split_into_words(constitution_text))
print(all_counts.most_common(10))
# [('the', 411), ('of', 287), ('shall', 191), ('and', 189), ('be', 125),
#  ('to', 114), ('in', 88), ('states', 82), ('or', 79), ('united', 55)]

> **Teaching note:** Goal: show that *unfiltered* counts are dominated by function words
> ("the", "of", "and") — nearly identical across any English text, so they reveal almost nothing.
> Motivates the stopword filter. "shall" sneaking into the raw top 5 is a nice preview that some
> high-frequency words ARE meaningful (legal obligation).

**A3 — Keeping only the meaningful words.**

In [ ]:
# A3 — solution
words = split_into_words(constitution_text)
meaningful = [w for w in words if w and w not in stopwords]
print(Counter(meaningful).most_common(10))
# [('shall', 191), ('states', 82), ('united', 55), ('state', 48), ('president', 34),
#  ('may', 33), ('congress', 29), ('house', 23), ('law', 23), ('section', 22)]

> **Teaching note:** Goal: the `if w and w not in stopwords` filter — the heart of term
> frequency. Now the top words actually describe the document (a structure of states, president,
> congress, law). Critical hook (Classification Logic): the stopword list is an *authored* filter —
> deciding "shall" stays but "should" goes is an editorial judgment, not a neutral fact.

**A4 — An edge case: predict, then run.**

In [ ]:
# A4 — solution
comment_words = [w for w in split_into_words(comments_text) if w and w not in stopwords]
comment_counts = Counter(comment_words)
print(comment_counts.most_common(0))
# []  -- most_common(0) asks for ZERO items, so you get an empty list (NOT an error, NOT "all").

> **Teaching note:** Goal: predict-then-run, like HW1 A4 — but here the surprise is a valid
> *empty* result rather than an error. Common wrong predictions: "it returns everything" or "it
> crashes." Discussion: an empty result is still data; in research, "we found zero" is a finding,
> not a failure. (`comment_counts` is defined here and reused in B2.)

**A5 — Wrap it in a reusable function.**

In [ ]:
# A5 — solution
def top_meaningful_words(text, n):
    words = split_into_words(text)
    meaningful = [w for w in words if w and w not in stopwords]
    return Counter(meaningful).most_common(n)

print(top_meaningful_words(constitution_text, 5))
# [('shall', 191), ('states', 82), ('united', 55), ('state', 48), ('president', 34)]

> **Teaching note:** Goal: refactor A3's logic into one reusable tool — the backbone of the
> rest of the homework (and HW3). Reinforce that a function is "write the recipe once, run it on
> any text." Students who hard-coded `constitution_text` inside the function should generalize it
> to the `text` parameter. The student note that follows makes the broader point — term frequency
> is prefab code nobody writes from scratch (Stack Overflow long before AI); we build it by hand
> so they can *read and judge* a borrowed/AI version later. Good spot to preview the **Day 7**
> AI-Agency lesson and set the norm that borrowing code is expected, not cheating.

## Part B — Whose words win? (4 exercises)

**B1 — Top 20 of each corpus, side by side.**

In [ ]:
# B1 — solution
print("CONSTITUTION:", top_meaningful_words(constitution_text, 20))
print("COMMENTS:", top_meaningful_words(comments_text, 20))
# CONSTITUTION -> shall, states, united, state, president, may, congress, house, law, section, ...
# COMMENTS     -> commandments, religion, ten, god, 10, children, schools, state, people, want, ...

> **Teaching note:** Goal: reuse the A5 function twice — the payoff of writing it. The contrast
> is the whole lesson: a vocabulary of governance (shall/states/president) vs. a vocabulary of
> religion-in-schools (commandments/god/children/schools). "state" appears in BOTH — a good bridge
> word to point out.

**B2 — A direct lookup: does the public speak the Constitution's language?**

In [ ]:
# B2 — solution
print("constitution:", comment_counts["constitution"])   # 8
print("commandments:", comment_counts["commandments"])   # 25

> **Teaching note:** Goal: indexing a Counter for ONE exact word (close reading) against the
> ranked totals (distant reading). The commenters DO invoke "constitution" (8×) — the top comment is
> literally "how about putting the constitution in classrooms?" — but their own keyword "commandments"
> (25×) wins 3-to-1. A missing key returns `0`, not an error — worth demonstrating live.

**B3 — Distinctive words: what one text says that the other never does.**

In [ ]:
# B3 — solution
con_set = {w for w in split_into_words(constitution_text) if w and w not in stopwords}
distinctive = [w for w, _ in top_meaningful_words(comments_text, 20) if w not in con_set]
print(distinctive)
# ['commandments', 'religion', 'god', 'children', 'schools', 'want', 'school', 'bible', 'follow', 'church', 'would']
# ('state'/'law'/'people'/'ten'/'one' drop out of the top-20 — they DO appear in the Constitution)

> **Teaching note:** Goal: set membership for "distinctive" vocabulary — the words the public
> argues about that the founding text is simply silent on (religion, god, children, schools). Critical
> hook: term frequency makes the *gap* visible — the debate is happening in a vocabulary the
> Constitution never uses, which is itself the commenters' point about church and state.

**B4 — Interpretation (model answer).**

> **Model answer (accept reasonable variants; look for specific counts + a named limitation):**
> Term frequency shows the comments are dominated by *commandments* (25) and *religion* (20), while the
> Constitution is dominated by *shall* (191) and *states* (82) — so the public is arguing about religion
> in a document that never uses the word. The commenters invoke *constitution* (8×) but their own
> keyword wins 3-to-1, so by the numbers the religious frame "wins" the comment thread. **What counting
> hides:** a raw tally can't tell *support* from *opposition* — many of those 25 "commandments" mentions
> are people arguing *against* the law, yet they look identical to endorsements in a frequency count.
> (That sentiment gap is exactly what HW3 takes up.)

> **Teaching note:** Reward any answer that (a) cites at least two specific word counts and (b) names a
> real limitation of frequency — most naturally that counts erase stance/sarcasm/context. This bridges
> directly to HW3 (VADER sentiment) and the close-vs-distant reading thread.

## Part C — Going further (2 exercises)

**C1 — Tuning your stopwords.**

In [ ]:
# C1 — solution
custom_stopwords = stopwords + ["10", "ten", "would", "people"]
words = [w for w in split_into_words(comments_text) if w and w not in custom_stopwords]
print(Counter(words).most_common(10))
# [('commandments', 25), ('religion', 20), ('god', 14), ('children', 11), ('schools', 10),
#  ('state', 10), ('want', 9), ('law', 9), ('constitution', 8), ('one', 8)]

> **Teaching note:** Goal: custom stopwords as an explicit, *visible* editorial act. Removing
> "10"/"ten" cleans the ranking — but note the cost: people deliberately said "the TEN commandments,"
> so filtering it slightly erases their emphasis. Good place to insist that every stopword choice be
> documentable, because it changes the result.

**C2 — Bring your own text (model).**

In [ ]:
# C2 — solution (any pasted corpus works; this stub just proves the call runs)
my_text = """
Keep church and state separate.
My kids should learn history not scripture.
Whose ten commandments though there are different versions.
"""
print(top_meaningful_words(my_text, 5))
# e.g. [('keep', 1), ('church', 1), ('state', 1), ('separate', 1), ('kids', 1)]

> **Teaching note:** Goal: prove the A5 function generalizes to *any* text students supply —
> the bridge to "bring your own cultural dataset" in the Week-2 workshop and the capstone. Remind
> students that real collection (scraping) is taught Day 8; hand-paste is fine now. Grade on a working
> call + a reflection, not on the corpus chosen.